In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

COUNTRIES = [
    ('Ethiopia', 'ethiopia.csv'),
    ('Kenya', 'kenya.csv'),
    ('Sudan', 'sudan.csv'),
    ('Tanzania', 'tanzania.csv'),
    ('Nigeria', 'nigeria.csv'),
]

NUMERIC_COLS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 
                'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']

OUTLIER_COLS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

OUTPUT_DIR = os.path.join('..', 'data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


In [2]:
# Store results for cross-country comparison
all_cleaned_dfs = []
country_stats = {}

for country_name, file_name in COUNTRIES:
    print(f"\n{'='*70}")
    print(f"PROCESSING: {country_name}")
    print(f"{'='*70}")
    
    # --- LOAD DATA ---
    path = os.path.join('..', file_name)
    
    if not os.path.exists(path):
        print(f"File not found: {path}. Skipping {country_name}.")
        continue
    
    df = pd.read_csv(path)
    df['Country'] = country_name
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    
    print(f"Data loaded: {len(df)} rows")
    print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
    
    # --- REPLACE SENTINEL VALUES ---
    df.replace(-999, np.nan, inplace=True)
    print("Replaced -999 sentinel values with NaN")
    
    # --- DUPLICATE CHECK ---
    dup_count = df.duplicated().sum()
    print(f"Duplicate rows found: {dup_count}")
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
        print(f"Duplicates removed. New shape: {df.shape}")
    
    # --- MISSING VALUE REPORT ---
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    missing_report = pd.DataFrame({
        'Missing Count': missing_count,
        'Percentage': missing_pct.round(2)
    })
    missing_report = missing_report[missing_report['Missing Count'] > 0].sort_values('Percentage', ascending=False)
    
    if len(missing_report) > 0:
        print("Missing Value Report:")
        print(missing_report)
        
        high_missing = missing_report[missing_report['Percentage'] > 5]
        if len(high_missing) > 0:
            print("Columns with >5% missing values:")
            for col in high_missing.index:
                print(f"  - {col}: {high_missing.loc[col, 'Percentage']:.1f}%")
    
    # --- DESCRIPTIVE STATISTICS ---
    print("Descriptive Statistics:")
    print(df[NUMERIC_COLS].describe().round(3))
    
    # --- OUTLIER DETECTION ---
    z_scores = np.abs(stats.zscore(df[OUTLIER_COLS].dropna()))
    outlier_mask = (z_scores > 3).any(axis=1)
    outlier_count = outlier_mask.sum()
    outlier_pct = (outlier_count / len(df)) * 100
    
    print(f"Outlier Detection (|Z| > 3): {outlier_count} rows ({outlier_pct:.1f}%)")
    
    # Cap outliers at 3 standard deviations
    for col in OUTLIER_COLS:
        mean_val = df[col].mean()
        std_val = df[col].std()
        upper_bound = mean_val + 3 * std_val
        lower_bound = mean_val - 3 * std_val
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    
    # --- HANDLE MISSING VALUES ---
    df[OUTLIER_COLS] = df[OUTLIER_COLS].ffill()
    
    # Drop rows with >30% missing values
    row_missing_pct = df.isna().sum(axis=1) / len(df.columns)
    rows_to_drop = (row_missing_pct > 0.3).sum()
    if rows_to_drop > 0:
        df = df[row_missing_pct <= 0.3]
        print(f"Dropped {rows_to_drop} rows with >30% missing values")
    
    print(f"Final cleaned shape: {df.shape}")
    
    # --- SAVE CLEANED DATA ---
    output_file = os.path.join(OUTPUT_DIR, f"{country_name.lower()}_clean.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")
    
    # Store for later use
    all_cleaned_dfs.append(df)
    country_stats[country_name] = {
        'mean_temp': df['T2M'].mean(),
        'std_temp': df['T2M'].std(),
        'mean_rain': df['PRECTOTCORR'].mean(),
        'std_rain': df['PRECTOTCORR'].std(),
        'extreme_heat_days': (df['T2M_MAX'] > 35).sum(),
        'outlier_pct': outlier_pct
    }

print(f"\n{'='*70}")
print(f"ALL COUNTRIES PROCESSED: {len(all_cleaned_dfs)} datasets saved")


PROCESSING: Ethiopia
File not found: ..\ethiopia.csv. Skipping Ethiopia.

PROCESSING: Kenya
File not found: ..\kenya.csv. Skipping Kenya.

PROCESSING: Sudan
File not found: ..\sudan.csv. Skipping Sudan.

PROCESSING: Tanzania
File not found: ..\tanzania.csv. Skipping Tanzania.

PROCESSING: Nigeria
File not found: ..\nigeria.csv. Skipping Nigeria.

ALL COUNTRIES PROCESSED: 0 datasets saved


In [3]:
all_cleaned_dfs = []
country_stats = {}

for country_name, file_name in COUNTRIES:
    print(f"\n{'='*50}")
    print(f"PROCESSING: {country_name}")
    
    path = os.path.join('..', file_name)
    df = pd.read_csv(path)
    df['Country'] = country_name
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    
    df.replace(-999, np.nan, inplace=True)
    
    dup_count = df.duplicated().sum()
    print(f"Duplicates: {dup_count}")
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
    
    missing_pct = (df.isna().sum() / len(df)) * 100
    high_missing = missing_pct[missing_pct > 5]
    if len(high_missing) > 0:
        print(f"Columns >5% missing: {list(high_missing.index)}")
    
    print(df[NUMERIC_COLS].describe().round(2))
    
    z_scores = np.abs(stats.zscore(df[OUTLIER_COLS].dropna()))
    outlier_count = (z_scores > 3).any(axis=1).sum()
    print(f"Outliers flagged: {outlier_count}")
    
    for col in OUTLIER_COLS:
        mean_val = df[col].mean()
        std_val = df[col].std()
        df[col] = df[col].clip(lower=mean_val - 3*std_val, upper=mean_val + 3*std_val)
    
    df[OUTLIER_COLS] = df[OUTLIER_COLS].ffill()
    row_missing = df.isna().sum(axis=1) / len(df.columns)
    df = df[row_missing <= 0.3]
    
    output_file = os.path.join(OUTPUT_DIR, f"{country_name.lower()}_clean.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")
    
    all_cleaned_dfs.append(df)
    country_stats[country_name] = {
        'mean_temp': df['T2M'].mean(),
        'std_temp': df['T2M'].std(),
        'mean_rain': df['PRECTOTCORR'].mean(),
        'std_rain': df['PRECTOTCORR'].std(),
        'extreme_heat_days': (df['T2M_MAX'] > 35).sum()
    }

print(f"\nDone! {len(all_cleaned_dfs)} countries processed.")

all_cleaned_dfs = []
country_stats = {}

for country_name, file_name in COUNTRIES:
    print(f"\n{'='*50}")
    print(f"PROCESSING: {country_name}")
    
    path = os.path.join('..', file_name)
    df = pd.read_csv(path)
    df['Country'] = country_name
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    
    df.replace(-999, np.nan, inplace=True)
    
    dup_count = df.duplicated().sum()
    print(f"Duplicates: {dup_count}")
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
    
    missing_pct = (df.isna().sum() / len(df)) * 100
    high_missing = missing_pct[missing_pct > 5]
    if len(high_missing) > 0:
        print(f"Columns >5% missing: {list(high_missing.index)}")
    
    print(df[NUMERIC_COLS].describe().round(2))
    
    z_scores = np.abs(stats.zscore(df[OUTLIER_COLS].dropna()))
    outlier_count = (z_scores > 3).any(axis=1).sum()
    print(f"Outliers flagged: {outlier_count}")
    
    for col in OUTLIER_COLS:
        mean_val = df[col].mean()
        std_val = df[col].std()
        df[col] = df[col].clip(lower=mean_val - 3*std_val, upper=mean_val + 3*std_val)
    
    df[OUTLIER_COLS] = df[OUTLIER_COLS].ffill()
    row_missing = df.isna().sum(axis=1) / len(df.columns)
    df = df[row_missing <= 0.3]
    
    output_file = os.path.join(OUTPUT_DIR, f"{country_name.lower()}_clean.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")
    
    all_cleaned_dfs.append(df)
    country_stats[country_name] = {
        'mean_temp': df['T2M'].mean(),
        'std_temp': df['T2M'].std(),
        'mean_rain': df['PRECTOTCORR'].mean(),
        'std_rain': df['PRECTOTCORR'].std(),
        'extreme_heat_days': (df['T2M_MAX'] > 35).sum()
    }

print(f"\nDone! {len(all_cleaned_dfs)} countries processed.")


PROCESSING: Ethiopia


FileNotFoundError: [Errno 2] No such file or directory: '..\\ethiopia.csv'

In [ ]:
all_cleaned_dfs = []
country_stats = {}

for country_name, file_name in COUNTRIES:
    print(f"\n{'='*50}")
    print(f"PROCESSING: {country_name}")
    
    path = os.path.join('..', file_name)
    df = pd.read_csv(path)
    df['Country'] = country_name
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    
    df.replace(-999, np.nan, inplace=True)
    
    dup_count = df.duplicated().sum()
    print(f"Duplicates: {dup_count}")
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
    
    missing_pct = (df.isna().sum() / len(df)) * 100
    high_missing = missing_pct[missing_pct > 5]
    if len(high_missing) > 0:
        print(f"Columns >5% missing: {list(high_missing.index)}")
    
    z_scores = np.abs(stats.zscore(df[OUTLIER_COLS].dropna()))
    outlier_count = (z_scores > 3).any(axis=1).sum()
    print(f"Outliers: {outlier_count}")
    
    for col in OUTLIER_COLS:
        mean_val = df[col].mean()
        std_val = df[col].std()
        df[col] = df[col].clip(lower=mean_val - 3*std_val, upper=mean_val + 3*std_val)
    
    df[OUTLIER_COLS] = df[OUTLIER_COLS].ffill()
    row_missing = df.isna().sum(axis=1) / len(df.columns)
    df = df[row_missing <= 0.3]
    
    output_file = os.path.join(OUTPUT_DIR, f"{country_name.lower()}_clean.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")
    
    all_cleaned_dfs.append(df)
    country_stats[country_name] = {
        'mean_temp': df['T2M'].mean(),
        'std_temp': df['T2M'].std(),
        'mean_rain': df['PRECTOTCORR'].mean(),
        'std_rain': df['PRECTOTCORR'].std(),
        'extreme_heat_days': (df['T2M_MAX'] > 35).sum()
    }

print(f"\nDone! {len(all_cleaned_dfs)} countries processed.")

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(16, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    monthly = df.set_index('Date').resample('M')['PRECTOTCORR'].sum()
    axes[idx].bar(monthly.index, monthly.values, color='steelblue', width=20)
    axes[idx].set_title(f'Monthly Total Precipitation — {df["Country"].iloc[0]}', fontweight='bold')
    axes[idx].set_ylabel('Precipitation (mm)')
    axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].axhline(y=monthly.mean(), color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6*len(all_cleaned_dfs), 5))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    corr = df[NUMERIC_COLS].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=axes[idx])
    axes[idx].set_title(df['Country'].iloc[0])

plt.suptitle('Correlation Matrices', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 2, figsize=(14, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = axes.reshape(1, -1)

for idx, df in enumerate(all_cleaned_dfs):
    axes[idx, 0].scatter(df['T2M'], df['RH2M'], alpha=0.3, s=10, c='teal')
    axes[idx, 0].set_xlabel('T2M (°C)')
    axes[idx, 0].set_ylabel('RH2M (%)')
    axes[idx, 0].set_title(f'T2M vs RH2M — {df["Country"].iloc[0]}')
    axes[idx, 0].grid(True, alpha=0.3)
    
    axes[idx, 1].scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, s=10, c='coral')
    axes[idx, 1].set_xlabel('T2M_RANGE (°C)')
    axes[idx, 1].set_ylabel('WS2M (m/s)')
    axes[idx, 1].set_title(f'T2M_RANGE vs WS2M — {df["Country"].iloc[0]}')
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(12, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    axes[idx].hist(df['PRECTOTCORR'].dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[idx].set_yscale('log')
    axes[idx].set_xlabel('Precipitation (mm/day)')
    axes[idx].set_title(f'Precipitation Distribution — {df["Country"].iloc[0]}')
    axes[idx].axvline(df['PRECTOTCORR'].mean(), color='red', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6*len(all_cleaned_dfs), 5))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    sample = df.sample(min(1000, len(df)))
    axes[idx].scatter(sample['T2M'], sample['RH2M'], s=sample['PRECTOTCORR']*20, 
                      c=sample['Month'], cmap='viridis', alpha=0.5)
    axes[idx].set_xlabel('T2M (°C)')
    axes[idx].set_ylabel('RH2M (%)')
    axes[idx].set_title(df['Country'].iloc[0])

plt.suptitle('T2M vs RH2M (Bubble size = Precipitation)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(16, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    monthly = df.set_index('Date').resample('M')['T2M'].mean()
    axes[idx].plot(monthly.index, monthly.values, color='crimson', marker='.', linewidth=1.5)
    axes[idx].set_title(f'Monthly Mean Temperature — {df["Country"].iloc[0]}', fontweight='bold')
    axes[idx].set_ylabel('Temperature (°C)')
    axes[idx].grid(True, alpha=0.3)
    
    x = np.arange(len(monthly))
    z = np.polyfit(x, monthly.values, 1)
    p = np.poly1d(z)
    axes[idx].plot(monthly.index, p(x), '--', color='gray', alpha=0.7, label=f'Trend: {z[0]*12:.3f}°C/yr')
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(16, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    monthly = df.set_index('Date').resample('M')['PRECTOTCORR'].sum()
    axes[idx].bar(monthly.index, monthly.values, color='steelblue', width=20)
    axes[idx].set_title(f'Monthly Total Precipitation — {df["Country"].iloc[0]}', fontweight='bold')
    axes[idx].set_ylabel('Precipitation (mm)')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6*len(all_cleaned_dfs), 5))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    corr = df[NUMERIC_COLS].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=axes[idx])
    axes[idx].set_title(df['Country'].iloc[0])

plt.suptitle('Correlation Matrices', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 2, figsize=(14, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = axes.reshape(1, -1)

for idx, df in enumerate(all_cleaned_dfs):
    axes[idx, 0].scatter(df['T2M'], df['RH2M'], alpha=0.3, s=10, c='teal')
    axes[idx, 0].set_xlabel('T2M (°C)')
    axes[idx, 0].set_ylabel('RH2M (%)')
    axes[idx, 0].set_title(f'T2M vs RH2M — {df["Country"].iloc[0]}')
    
    axes[idx, 1].scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, s=10, c='coral')
    axes[idx, 1].set_xlabel('T2M_RANGE (°C)')
    axes[idx, 1].set_ylabel('WS2M (m/s)')
    axes[idx, 1].set_title(f'T2M_RANGE vs WS2M — {df["Country"].iloc[0]}')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(12, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    axes[idx].hist(df['PRECTOTCORR'].dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[idx].set_yscale('log')
    axes[idx].set_xlabel('Precipitation (mm/day)')
    axes[idx].set_title(f'Precipitation Distribution — {df["Country"].iloc[0]}')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6*len(all_cleaned_dfs), 5))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    sample = df.sample(min(1000, len(df)))
    axes[idx].scatter(sample['T2M'], sample['RH2M'], s=sample['PRECTOTCORR']*20, 
                      c=sample['Month'], cmap='viridis', alpha=0.5)
    axes[idx].set_xlabel('T2M (°C)')
    axes[idx].set_ylabel('RH2M (%)')
    axes[idx].set_title(df['Country'].iloc[0])

plt.suptitle('T2M vs RH2M (Bubble size = Precipitation)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
### Correlation Analysis Interpretation

**Recurring Strong Correlations:**

1. **T2M_MAX vs T2M (r ≈ 0.9+)**: Expected relationship — mean and maximum temperatures are highly correlated.

2. **T2M vs RH2M (strongly negative)**: Warmer temperatures drive lower relative humidity. This amplifies drought stress on vegetation.

3. **T2M_RANGE vs RH2M**: Larger daily temperature swings are associated with drier air.

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 2, figsize=(14, 4.5 * len(all_cleaned_dfs)))

if len(all_cleaned_dfs) == 1:
    axes = axes.reshape(1, -1)

for idx, df in enumerate(all_cleaned_dfs):
    country_name = df['Country'].iloc[0]
    
    # T2M vs RH2M
    axes[idx, 0].scatter(df['T2M'], df['RH2M'], alpha=0.3, c='teal', edgecolors='none', s=10)
    axes[idx, 0].set_xlabel('T2M (C)')
    axes[idx, 0].set_ylabel('RH2M (%)')
    axes[idx, 0].set_title(f'T2M vs RH2M — {country_name}')
    axes[idx, 0].grid(True, alpha=0.3)
    
    corr_val = df['T2M'].corr(df['RH2M'])
    axes[idx, 0].text(0.05, 0.95, f'r = {corr_val:.3f}', transform=axes[idx, 0].transAxes,
                      fontsize=11, verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # T2M_RANGE vs WS2M
    axes[idx, 1].scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, c='coral', edgecolors='none', s=10)
    axes[idx, 1].set_xlabel('T2M_RANGE (C)')
    axes[idx, 1].set_ylabel('WS2M (m/s)')
    axes[idx, 1].set_title(f'T2M_RANGE vs WS2M — {country_name}')
    axes[idx, 1].grid(True, alpha=0.3)
    
    corr_val2 = df['T2M_RANGE'].corr(df['WS2M'])
    axes[idx, 1].text(0.05, 0.95, f'r = {corr_val2:.3f}', transform=axes[idx, 1].transAxes,
                      fontsize=11, verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(12, 4 * len(all_cleaned_dfs)))

if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    country_name = df['Country'].iloc[0]
    
    axes[idx].hist(df['PRECTOTCORR'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.7)
    axes[idx].set_yscale('log')
    axes[idx].set_xlabel('Precipitation (mm/day)')
    axes[idx].set_ylabel('Frequency (log scale)')
    axes[idx].set_title(f'Precipitation Distribution (Log Scale) — {country_name}')
    axes[idx].axvline(df['PRECTOTCORR'].mean(), color='red', linestyle='--', 
                      label=f"Mean: {df['PRECTOTCORR'].mean():.2f}")
    axes[idx].legend()

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6 * len(all_cleaned_dfs), 5))

if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    country_name = df['Country'].iloc[0]
    
    sample_df = df.sample(min(1000, len(df)))
    
    scatter = axes[idx].scatter(
        sample_df['T2M'], 
        sample_df['RH2M'], 
        s=sample_df['PRECTOTCORR'] * 20,
        c=sample_df['Month'],
        cmap='viridis',
        alpha=0.5,
        edgecolors='white',
        linewidth=0.5
    )
    axes[idx].set_xlabel('T2M (C)')
    axes[idx].set_ylabel('RH2M (%)')
    axes[idx].set_title(f'{country_name}')
    axes[idx].grid(True, alpha=0.3)

plt.colorbar(scatter, ax=axes, label='Month')
plt.suptitle('Temperature vs Humidity (Bubble size = Precipitation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()